# End-to-End BYOC (Bring Your Own Chunks) with Visual Grounding

This notebook demonstrates the complete process for successfully ingesting custom chunks with associated images and retrieving them in Vertex AI Search.

**Process:**
1.  **Create a Vertex AI Search Data Store**: To hold our unstructured data.
2.  **Create a Search App (Engine)**: To provide the search and answer generation capabilities.
3.  **Ingest a BYOC Document**: The core step, showing how to map `blobAssets` (images) directly to specific chunks using `chunkFields`.
4.  **Query the App**: Use `streamAnswer` to retrieve both the custom chunk content and the associated image (`blobAttachments`).

### Step 0: Install Dependencies & Authenticate

In [1]:
!pip install google-cloud-discoveryengine requests google-auth -q

In [2]:
import time
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions

# --- CONFIGURATION ---
# Please replace these with your actual project details.
PROJECT_ID = "gemini-ent-agent-demos"
LOCATION = "global"  # Or "us", "eu"
DATA_STORE_ID = f"byoc-image-guide-ds-{int(time.time())}"  # Unique ID for the Data Store
ENGINE_ID = f"byoc-image-guide-app-{int(time.time())}"     # Unique ID for the Search App

# Set up client options for the specified location
client_options = (
    ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
    if LOCATION != "global"
    else None
)

print(f"Using Project ID: {PROJECT_ID}")
print(f"Data Store ID: {DATA_STORE_ID}")
print(f"Engine ID: {ENGINE_ID}")



Using Project ID: gemini-ent-agent-demos
Data Store ID: byoc-image-guide-ds-1776710150
Engine ID: byoc-image-guide-app-1776710150


### Step 1: Create the Data Store

We will create an Unstructured Data Store designed for search solutions.


In [3]:
def create_data_store():
    """Creates a new data store."""
    ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    data_store = discoveryengine.DataStore(
        display_name="BYOC Image Demo Store",
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )

    print(f"Creating Data Store: {DATA_STORE_ID}...")
    try:
        request = discoveryengine.CreateDataStoreRequest(
            parent=parent,
            data_store_id=DATA_STORE_ID,
            data_store=data_store,
        )
        operation = ds_client.create_data_store(request=request)
        response = operation.result()
        print(f"Data Store created successfully: {response.name}")
        return response
    except Exception as e:
        print(f"Error creating data store: {e}")
        print("This may happen if the data store already exists. Please check the console.")


create_data_store()


Creating Data Store: byoc-image-guide-ds-1776710150...
Data Store created successfully: projects/473900759553/locations/global/collections/default_collection/dataStores/byoc-image-guide-ds-1776710150


name: "projects/473900759553/locations/global/collections/default_collection/dataStores/byoc-image-guide-ds-1776710150"
display_name: "BYOC Image Demo Store"
industry_vertical: GENERIC
solution_types: SOLUTION_TYPE_SEARCH
content_config: CONTENT_REQUIRED
default_schema_id: "default_schema"
document_processing_config {
  name: "projects/473900759553/locations/global/collections/default_collection/dataStores/byoc-image-guide-ds-1776710150/documentProcessingConfig"
  default_parsing_config {
    digital_parsing_config {
    }
  }
}

### Step 2: Create the Search App (Engine)

Next, we create a Search App with LLM add-ons enabled (required for `streamAnswer`) and link it to our new Data Store.


In [4]:
def create_engine():
    """Creates a new search app."""
    engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    engine = discoveryengine.Engine(
        display_name="BYOC Image Demo App",
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        data_store_ids=[DATA_STORE_ID],
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
            search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM],
        ),
    )

    print(f"Creating Engine (App): {ENGINE_ID}...")
    try:
        request = discoveryengine.CreateEngineRequest(
            parent=parent,
            engine_id=ENGINE_ID,
            engine=engine,
        )
        operation = engine_client.create_engine(request=request)
        response = operation.result()
        print(f"Engine created successfully: {response.name}")
        return response
    except Exception as e:
        print(f"Error creating engine: {e}")
        print("This may happen if the engine already exists. Please check the console.")


create_engine()

# Wait for a brief period to ensure the backend is fully provisioned before importing documents.
print("\nWaiting 20 seconds for provisioning to complete...")
time.sleep(20)


Creating Engine (App): byoc-image-guide-app-1776710150...
Engine created successfully: projects/473900759553/locations/global/collections/default_collection/engines/byoc-image-guide-app-1776710150

Waiting 20 seconds for provisioning to complete...


### Step 3: Ingest the BYOC Document with Image Mapping

This is the most critical step. We will construct a document payload that contains:
1.  `blob_assets`: The raw base64-encoded image data.
2.  `chunked_document`: Our custom-defined chunks.
3.  **`chunk_fields`**: The explicit mapping that tells Vertex AI Search which image (`blobAssetId`) belongs to which chunk.


In [5]:
import json
import time
import base64
import requests
import google.auth
import google.auth.transport.requests

def ingest_byoc_document_v3():
    """
    Ingests a BYOC document using the REST API with the v3 format.
    
    Key changes for v3 API:
    1. The 'content' block uses mimeType "application/json" instead of "text/plain".
    2. The content.rawBytes contains the base64-encoded Canonical Doc JSON (not a dummy value).
    3. jsonData contains the stringified version of the same Canonical Doc JSON.
    """

    # 1. Automatically get the GCP access token
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    # 2. Construct the Endpoint URL
    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/dataStores/{DATA_STORE_ID}/branches/default_branch/documents:import"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    # 1x1 transparent PNG base64 string
    encoded_image_b64 = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkYAAAAAYAAjCB0C8AAAAASUVORK5CYII="

    # 3. Build the Canonical Doc JSON (v3 format)
    # This is the core document payload that includes layout, chunks, and blob assets.
    canonical_doc_json = {
        "title": "Chemical Price Volatility Analysis",
        "description": "Analysis of price volatility for Ethylene, Caustic soda, and Benzene from 2000 to 2019.",
        "blobAssets": [
            {
                "name": "blob_1",
                "content": encoded_image_b64,
                "mimeType": "image/png"
            }
        ],
        "chunkedDocument": {
            "chunks": [
                {
                    "chunkId": "custom_chunk_with_image_1",
                    "content": "This chunk describes the price volatility of Ethylene, Caustic soda, and Benzene from 2000 to 2019.",
                    "chunkFields": [
                        {
                            "name": "chart_image",
                            # CRITICAL STEP: Explicitly link the blobAssetId here
                            "imageChunkField": {
                                "blobAssetId": "blob_1"
                            }
                        }
                    ]
                }
            ]
        }
    }

    # Stringify the Canonical Doc JSON
    canonical_doc_json_str = json.dumps(canonical_doc_json)
    
    # Base64 encode the Canonical Doc JSON for content.rawBytes
    canonical_doc_json_b64 = base64.b64encode(canonical_doc_json_str.encode('utf-8')).decode('utf-8')

    # 4. Construct the final Discovery Engine payload using v3 format
    payload = {
        "inlineSource": {
            "documents": [
                {
                    "id": "byoc_doc_with_mapped_image_v3",

                    # A. Pass the Canonical Doc JSON as a stringified JSON
                    "jsonData": canonical_doc_json_str,

                    # B. Use mimeType "application/json" and base64 encode the 
                    #    Canonical Doc JSON as rawBytes (v3 format)
                    "content": {
                        "mimeType": "application/json",
                        "rawBytes": canonical_doc_json_b64
                    }
                }
            ]
        }
    }

    # 5. Make the POST request
    print(f"Ingesting BYOC document via REST API (v3 format) to {base_url}...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        print("\n[SUCCESS] Import operation initiated successfully!")
        print("Response:", json.dumps(response.json(), indent=2))
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

    return response

# Execute the ingestion
ingest_byoc_document_v3()

# Wait for indexing
print("\nWaiting 5 minutes for indexing to complete before querying...")
time.sleep(300)


### Step 4: Query the App and Verify Blob Attachments

Finally, we query the app using `streamAnswer` and inspect the response. We expect to see the `blobAttachments` field populated in the citation source because we correctly mapped the image to the chunk during ingestion.


In [16]:
import json
import requests
import google.auth
import google.auth.transport.requests

def query_answer_rest():
    """Queries the app using the REST API to bypass Python SDK limitations."""

    # 1. Automatically get the GCP access token
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    # 2. Construct the Endpoint URL
    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_serving_config:answer"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    query_text = "price volatility of Ethylene, Caustic soda, and Benzene from 2000 to 2019"

    # 3. Construct the JSON Payload exactly as the customer's curl command
    payload = {
        "query": { "text": query_text },
        "answerGenerationSpec": {
            "includeCitations": True,
            "multimodalSpec": {"imageSource": "CORPUS_IMAGE_ONLY"},
            "modelSpec": { "modelVersion": "stable" }
        }
    }

    print(f"\nSending answer query: '{query_text}'...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        answer = data.get("answer", {})
        citations = answer.get("citations", [])

        found_attachments = False

        # 4. Parse the response for Citations and Blob Attachments
        for citation in citations:
            for source in citation.get("sources", []):
                chunk_info = source.get("chunkInfo", {})

                print("\n--- Citation Source Found ---")
                print(f"Custom Chunk Content: '{chunk_info.get('content', '')}'")

                # Check for blob attachments in the chunk info
                blob_attachments = chunk_info.get("blobAttachments", [])

                if blob_attachments:
                    found_attachments = True
                    print(f"\n[SUCCESS] Found {len(blob_attachments)} Blob Attachment(s)!")
                    for blob in blob_attachments:
                        print(f" -> Blob Name: {blob.get('name')}")
                        print(f" -> MIME Type: {blob.get('mimeType')}")
                        print(f" -> Base64 Content Length: {len(blob.get('content', ''))}")
                else:
                    print("\n[WARNING] No Blob Attachments were returned in this source.")

        if not found_attachments:
            print("\n--- Final Result ---")
            print("Query completed, but no blob attachments were found.")
            print("This could be because indexing is not yet complete. Please try running this cell again in a few minutes.")

    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

# Run the REST query
query_answer_rest()



Sending answer query: 'price volatility of Ethylene, Caustic soda, and Benzene from 2000 to 2019'...

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Citation Source Found ---
Custom Chunk Content: ''

[WARNING] No Blob Attachments were returned in this source.

--- Final Result ---
Query completed, but no blob attachments were found.
This could be because indexing is not yet complete. Please try running this cell again in a few minutes.
